# AI Herding · Next Step 3: Controlled prompt responses and one-round replay

**A Jupyter notebook with 9 code cells, reusing the directories and environment from Steps 1 and 2.**

The planned third step varies market-maker prompts and their resulting outputs while fixing other agents' responses and the order-processing sequence. Reuse the Current, Earliest public, and Mandatory two-sided prompts; no additional prompt condition is required.

| Mode | Procedure | New model API calls |
|---|---|---|
| `cached_orders` (default) | Freeze 6 non-market-makers and replay existing market-maker orders from the three conditions; check controls and matching | **0** |
| `new_mm_api` | Generate fresh responses from two market makers under the three prompts, then replay them in the same fixed environment | **Required**; 18 planned requests by default |

**The default mode is preparatory/supplementary replay and must not be described as a new model prompt-response experiment.** To run a new controlled prompt-response experiment, set `RUN_MODE` to `"new_mm_api"` in cell 1 and supply a usable team/personal model API configuration.

This file contains no credentials. Default mode neither reads a key nor generates new model responses. API mode calls only the two market makers; the other 6 agents always reuse the same cache. Response parsing, caching, and resumption were checked with a mock client; no real model requests were made before delivery.

The default model preserves the report's `gpt-4o-2024-11-20`. The official model page marks this snapshot Deprecated; access still depends on the account/endpoint. The script does not substitute models automatically. References: [GPT-4o model page](https://developers.openai.com/api/docs/models/gpt-4o) and [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=chat).

Code and evidence are pinned to team commit `074faff3c8f6255c917d2a2593a5b476c947ca84`, with Step 1's cap and Step 2's logging. Distributed example outputs are cleared; save the local results after execution.


## Code cell 1/9: Select the mode and locate Step 2

Check replay first with the default `cached_orders`. Selecting `new_mm_api` enables real API requests in cell 5.

Two market makers × three prompts × three response repetitions = **18 planned requests**. Matching completed response caches can be reused. Failed requests are not retried automatically. Installing Python dependencies is not a model API call.


In [ ]:
import sys,os,json,csv,hashlib,shutil,subprocess,ast,uuid
from pathlib import Path
from datetime import datetime,timezone
from collections import defaultdict,Counter
from html import escape

RUN_MODE = "cached_orders"  # Select "new_mm_api" for a fresh-response experiment; this calls the model API.
REPEATS = 3
API_MODEL = "gpt-4o-2024-11-20"
API_BASE_URL = ""  # Leave blank for the OpenAI endpoint, or enter the complete team-compatible endpoint URL.
MAX_NEW_API_REQUESTS = 18
STEP2_WORK_OVERRIDE = None  # Set the Step 2 directory path here if automatic discovery fails.

assert RUN_MODE in ("cached_orders","new_mm_api")
assert REPEATS in (1,2,3)
if sys.version_info < (3,11):raise RuntimeError("Use a Python 3.11 or later kernel.")
hint=STEP2_WORK_OVERRIDE or os.environ.get("AHE_STEP2_WORKDIR")
candidates=[Path(hint).expanduser()] if hint else [Path.cwd()/"AI_Herding_Step2_Work",Path.home()/"AI_Herding_Step2_Work"]
if not hint:candidates += [p/"AI_Herding_Step2_Work" for p in list(Path.cwd().parents)[:3]]
STEP2_WORK=next((p.resolve() for p in candidates if (p/"source_logged/src/order_lifecycle_audit.py").is_file()),None)
if STEP2_WORK is None:raise FileNotFoundError("Set STEP2_WORK_OVERRIDE to the Step 2 AI_Herding_Step2_Work directory.")
SOURCE_LOGGED=STEP2_WORK/"source_logged"
EXPECTED_SOURCE_HASHES={'src/market/orders/order_state_manager.py': '9ed4404e4c54f0dfd32c4bff1b047264fdaab19b2eb3a1c47b15f60679a28020', 'src/market/orders/order_service_factory.py': '3fa6ae07a599b36a9f095a207cb082b6681ad3817bf0d7f633e4959b428586de', 'src/base_sim.py': 'b5dc6d64012d372be01771ca90385db9a3da282e693c03e97ce2936ab28b379d', 'src/order_lifecycle_audit.py': '274539040db6008a230a21b9682c7cade7f87488eb6b316602f1599ad57cc634'}
for relative,expected in EXPECTED_SOURCE_HASHES.items():
    text=(SOURCE_LOGGED/relative).read_text(encoding="utf-8")
    if hashlib.sha256(text.encode()).hexdigest()!=expected:
        raise RuntimeError(f"Source from the previous steps differs from the verified version: {relative}. Inspect and preserve your changes first.")
STEP1_WORK=STEP2_WORK.parent/"AI_Herding_Step1_Work"
TEST_PYTHON=STEP1_WORK/"test_environment"/("Scripts/python.exe" if os.name=="nt" else "bin/python")
if not TEST_PYTHON.exists():raise FileNotFoundError("Step 1 test environment not found. Restore the working directories from the previous steps.")
WORK_DIR=STEP2_WORK.parent/"AI_Herding_Step3_Work"
WORK_DIR.mkdir(parents=True,exist_ok=True)
REPLAY_ROOT=WORK_DIR/"source_replay"
SESSION_DIR=WORK_DIR/("session_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:6])
SESSION_DIR.mkdir()
PROTOCOL_DIR=SESSION_DIR/"protocol"
PROTOCOL_DIR.mkdir()
print("Current mode: ",RUN_MODE)
print("New model calls for default replay: 0; requests planned for API mode: ",2*3*REPEATS)
print("Step 2 directory found: ",STEP2_WORK)
print("Session directory: ",SESSION_DIR)


## Code cell 2/9: Prepare dependencies and display helpers

Reuse the test environment from the previous steps. Check/install the OpenAI SDK only when API mode is explicitly selected.


In [ ]:
probe_source = r"""
import importlib, json, sys
needed=[]
for name,spec in [('numpy','numpy>=1.26'),('pandas','pandas>=2.2'),('pydantic','pydantic>=2.11,<3')]:
    try:importlib.import_module(name)
    except (ImportError,OSError):needed.append(spec)
if sys.argv[1]=='new_mm_api':
    try:import openai
    except ImportError:needed.append('openai>=1.86')
print(json.dumps(needed))
"""
probe=subprocess.run([str(TEST_PYTHON),"-c",probe_source,RUN_MODE],capture_output=True,text=True,encoding="utf-8",check=True)
needed=json.loads(probe.stdout)
if needed:
    install=subprocess.run([str(TEST_PYTHON),"-m","pip","install","--disable-pip-version-check",*needed],capture_output=True,text=True,encoding="utf-8")
    if install.returncode:raise RuntimeError(install.stdout[-2000:]+install.stderr[-3000:])

def read_csv(path):
    with Path(path).open(encoding="utf-8-sig",newline="") as stream:return list(csv.DictReader(stream))
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
def digest(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()).hexdigest()
def display_table(rows,columns):
    from IPython.display import display,HTML
    head="".join(f"<th>{escape(str(label))}</th>" for key,label in columns)
    body="".join("<tr>"+"".join(f"<td>{escape(str(row.get(key,'')))}</td>" for key,label in columns)+"</tr>" for row in rows)
    display(HTML("<table style='border-collapse:collapse' border='1' cellpadding='6'><tr>"+head+"</tr>"+body+"</table>"))
def child_environment():
    env=os.environ.copy();env['PYTHONIOENCODING']='utf-8';return env
print("Test environment ready.")


## Code cell 3/9: Write the complete replay and API-generation code

Run this long cell in full. Write the shared cache utilities, offline replay worker, and optional API generator only into the Step 3 copy. Keep the previous copies.


In [ ]:
NEW_FILES = {
'src/step3_replay_common.py': r"""'''Cache and protocol utilities for controlled one-round market-maker replays.'''
from __future__ import annotations
import csv
import hashlib
import json
import math
from collections import defaultdict
from copy import deepcopy
from pathlib import Path

ARMS = ('current', 'earliest_public', 'mandatory_two_sided')
MAKER_IDS = ('4', '5')
OTHER_IDS = ('0', '1', '2', '3', '6', '7')
AGENT_IDS = tuple(str(i) for i in range(8))
SOURCE_COMMIT = '074faff3c8f6255c917d2a2593a5b476c947ca84'


def canonical_value(value):
    if isinstance(value, dict):
        return {str(k): canonical_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [canonical_value(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return str(value)
    return value


def digest(value):
    body = json.dumps(canonical_value(value), ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(body.encode('utf-8')).hexdigest()


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(canonical_value(value), ensure_ascii=False, indent=2), encoding='utf-8')


def read_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def read_prompts(run):
    return {str(row['agent_id']): row for row in
            (json.loads(line) for line in (run / 'data/rendered_prompts.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())}


def reconstruct_orders(run):
    '''The CSV lacks replace_decision. Normalize Add for a new, empty one-round book.'''
    rows = read_csv(run / 'structured_decisions.csv')
    grouped = defaultdict(list)
    for row in rows:
        assert int(row['round']) == 0, 'Only the initial one-round probes are supported'
        grouped[str(row['agent_id'])].append(row)
    assert set(grouped) == set(AGENT_IDS)
    responses = {}
    for aid in AGENT_IDS:
        orders = []
        for row in grouped[aid]:
            side = row['decision'].strip().lower()
            if side == 'hold':
                continue
            if side not in ('buy', 'sell'):
                raise ValueError(f'Unexpected side: {side}')
            kind = row['order_type'].split('.')[-1].lower()
            if kind not in ('market', 'limit'):
                raise ValueError(f'Unexpected order type: {kind}')
            quantity = int(float(row['quantity']))
            price = float(row['price']) if kind == 'limit' else None
            assert quantity > 0 and (price is None or math.isfinite(price) and price > 0)
            orders.append({'decision': side.title(), 'quantity': quantity,
                           'order_type': kind, 'price_limit': price, 'stock_id': 'DEFAULT_STOCK'})
        first = grouped[aid][0]
        response = {
            'orders': orders, 'replace_decision': 'Add',
            'reasoning': first.get('reasoning', ''),
            'valuation': float(first['valuation']),
            'valuation_reasoning': first.get('valuation_reasoning', ''),
            'response_source': 'recorded_order_rows_normalized_for_empty_initial_book',
            'source_run': run.name,
        }
        responses[aid] = response
    return responses


def initial_control(parameters):
    agents = parameters['AGENT_PARAMS']
    return canonical_value({
        'initial_price': parameters['INITIAL_PRICE'],
        'fundamental_price': parameters['FUNDAMENTAL_PRICE'],
        'transaction_cost': parameters['TRANSACTION_COST'],
        'lendable_shares': parameters['LENDABLE_SHARES'],
        'agent_params': agents, 'num_rounds': 1,
        'initial_order_book': {'bids': [], 'asks': []},
    })


def prepare_bundle(source, output, *, model='gpt-4o-2024-11-20', base_url='', repeats=3, seed=42,
                   reference_run='prompt_probe_r1_current'):
    source, output = Path(source), Path(output)
    assert repeats in (1, 2, 3), 'The evidence package has three cached profiles per arm'
    reference = source / 'data/runs' / reference_run
    parameters = json.loads((reference / 'parameters.json').read_text(encoding='utf-8'))
    assert parameters['NUM_ROUNDS'] == 1 and parameters['INITIAL_PRICE'] == 56.0
    assert parameters['AGENT_PARAMS']['max_order_size'] == 1000
    assert parameters['AGENT_PARAMS']['agent_composition'] == {
        'default': 2, 'optimistic': 2, 'market_maker': 2, 'speculator': 2}
    assert parameters['LENDABLE_SHARES'] == 0 and parameters['TRANSACTION_COST'] == 0
    reference_responses = reconstruct_orders(reference)
    reference_prompts = read_prompts(reference)
    frozen = {aid: deepcopy(reference_responses[aid]) for aid in OTHER_IDS}
    controls = initial_control(parameters)
    variants, requests, sources = [], [], []
    system_prompts = {}
    for arm in ARMS:
        system_prompts[arm] = read_prompts(source / 'data/runs' / f'prompt_probe_r1_{arm}')['4']['system_prompt']
    for repeat in range(1, repeats + 1):
        for arm in ARMS:
            run = source / 'data/runs' / f'prompt_probe_r{repeat}_{arm}'
            run_parameters = json.loads((run / 'parameters.json').read_text(encoding='utf-8'))
            assert digest(initial_control(run_parameters)) == digest(controls)
            prompts = read_prompts(run)
            for aid in MAKER_IDS:
                assert prompts[aid]['user_prompt'] == reference_prompts[aid]['user_prompt']
                assert prompts[aid]['system_prompt'] == system_prompts[arm]
            original = reconstruct_orders(run)
            responses = {**deepcopy(frozen), **{aid: deepcopy(original[aid]) for aid in MAKER_IDS}}
            variants.append({
                'arm': arm, 'repeat': repeat, 'responses': responses,
                'target_source_run': run.name, 'source_system_prompt_sha256': digest(system_prompts[arm]),
                'target_response_sha256': digest({aid: responses[aid] for aid in MAKER_IDS}),
            })
            sources.append({'run_id': run.name,
                            'decisions_csv_sha256': hashlib.sha256((run / 'structured_decisions.csv').read_bytes()).hexdigest()})
            for aid in MAKER_IDS:
                request = {
                    'request_key': f'{arm}:r{repeat}:agent{aid}', 'arm': arm,
                    'repeat': repeat, 'agent_id': aid, 'model': model,
                    'api_base_url': base_url.rstrip('/') if base_url else 'https://api.openai.com/v1',
                    'temperature': 0.0, 'seed': seed + repeat - 1,
                    'max_completion_tokens': 2000,
                    'system_prompt': system_prompts[arm],
                    'user_prompt': reference_prompts[aid]['user_prompt'],
                    'response_schema': 'FeatureRegistry core schema, no optional features; single stock',
                }
                request['request_sha256'] = digest(request)
                requests.append(request)
    bundle = {
        'source_commit': SOURCE_COMMIT, 'source_mode': 'cached_orders',
        'reference_run': reference_run, 'initial_control': controls,
        'agent_processing_order': list(AGENT_IDS), 'maker_ids': list(MAKER_IDS),
        'other_ids': list(OTHER_IDS), 'frozen_other_responses': frozen,
        'frozen_other_response_sha256': digest(frozen),
        'initial_control_sha256': digest(controls),
        'repeats': repeats, 'variants': variants, 'source_records': sources,
        'model_for_optional_api': model,
        'normalization': {
            'replace_decision': 'Add',
            'reason': 'Historical CSV does not store Add/Cancel/Replace. A new empty initial book is assumed.',
            'scope': 'Recorded-order replay only; not exact historical full-response replay.',
        },
        'priority_rule': 'Fixed agent rank and within-agent order index; synthetic timestamps; no engine shuffling.',
        'order_size_rule': 'Reject requests above 1000 before resource commitment in every condition.',
        'evidence_scope': 'Conditional effect of cached target order profiles; no identified new prompt-generation effect.',
    }
    plan = {'model': model, 'requests': requests, 'planned_requests': len(requests),
            'api_base_url': base_url.rstrip('/') if base_url else 'https://api.openai.com/v1',
            'frozen_other_response_sha256': digest(frozen), 'source_commit': SOURCE_COMMIT,
            'schema_features': [], 'automated_retries': 0}
    save_json(output / 'cached_bundle.json', bundle)
    save_json(output / 'api_plan.json', plan)
    save_json(output / 'protocol_summary.json', {
        'source_mode': bundle['source_mode'], 'reference_run': reference_run,
        'variants': len(variants), 'frozen_agents': len(OTHER_IDS),
        'target_agents': len(MAKER_IDS), 'new_model_calls_for_cached_mode': 0,
        'planned_new_model_requests_for_api_mode': len(requests),
        'initial_control_sha256': digest(controls),
        'frozen_other_response_sha256': digest(frozen),
        'normalization': bundle['normalization'], 'order_size_rule': bundle['order_size_rule'],
    })
    return bundle, plan
""",
'scripts/run_step3_controlled_replay.py': r"""'''Prepare cached order profiles or replay them with frozen counterparties and priorities.'''
from __future__ import annotations
import argparse
import csv
import json
import logging
import random
import socket
import sys
from collections import Counter
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

parser=argparse.ArgumentParser()
parser.add_argument('phase',choices=('prepare','replay'))
parser.add_argument('--source',type=Path,required=True)
parser.add_argument('--output',type=Path,required=True)
parser.add_argument('--bundle',type=Path)
parser.add_argument('--model',default='gpt-4o-2024-11-20')
parser.add_argument('--base-url',default='')
parser.add_argument('--repeats',type=int,default=3)
args=parser.parse_args()
source=args.source.resolve();output=args.output.resolve()
output.mkdir(parents=True,exist_ok=True)
sys.dont_write_bytecode=True
sys.path.insert(0,str(source/'src'))
from step3_replay_common import prepare_bundle,digest,save_json,read_csv,ARMS,MAKER_IDS,OTHER_IDS,AGENT_IDS

network_attempts=[]
def deny_network(*unused_args,**unused_kwargs):
    network_attempts.append('blocked')
    raise RuntimeError('The replay worker cannot make network or model API requests')
socket.socket.connect=deny_network
socket.socket.connect_ex=deny_network
socket.create_connection=deny_network

if args.phase=='prepare':
    bundle,plan=prepare_bundle(source,output,model=args.model,base_url=args.base_url,repeats=args.repeats)
    print(json.dumps({'variants':len(bundle['variants']), 'frozen_agents':len(OTHER_IDS),
                      'target_agents':len(MAKER_IDS), 'optional_new_model_requests':plan['planned_requests'],
                      'model_api_calls':0,'network_attempts':len(network_attempts)},ensure_ascii=True))
    raise SystemExit(0)
if args.bundle is None:
    raise ValueError('--bundle is required for replay')
bundle=json.loads(args.bundle.read_text(encoding='utf-8'))
assert bundle['source_mode'] in ('cached_orders','new_mm_api')
assert bundle['initial_control']['initial_order_book']=={'bids':[],'asks':[]}
assert tuple(bundle['agent_processing_order'])==AGENT_IDS
assert digest(bundle['frozen_other_responses'])==bundle['frozen_other_response_sha256']

sys.path.insert(0,str(source/'tests'))
import _logging_stub
_logging_stub.install()
logging.disable(logging.CRITICAL)
from agents.base_agent import BaseAgent
from agents.agent_manager.agent_repository import AgentRepository
from agents.agent_manager.base_agent_manager import AgentManager
from agents.agent_manager.services.agent_decision_service import AgentDecisionService
from agents.agent_manager.services.borrowing_repository import BorrowingRepository
from market.orders.order_repository import OrderRepository
from market.orders.order_book import OrderBook
from market.orders.order_service_factory import OrderServiceFactory
from market.state.sim_context import SimulationContext
from market.engine.match_engine import MatchingEngine
from market.data_recorder import DataRecorder
from services.shared_service_factory import SharedServiceFactory
from order_lifecycle_audit import LifecycleAudit,write_csv,clean


class CachedResponseAgent(BaseAgent):
    def make_decision(self,market_state,history,round_number):
        assert round_number==0,'Only the initial one-round state is supported'
        self.replay_count+=1
        return deepcopy(self.response)


def replay_one(variant,verification=False):
    random.seed(42)
    controls=bundle['initial_control'];ap=controls['agent_params']
    assert not ap['leverage_params']['enabled'] and not ap['allow_short_selling']
    context=SimulationContext(1,float(controls['initial_price']),float(controls['fundamental_price']),
                              float(controls['fundamental_price']),float(controls['transaction_cost']))
    role_by_id={'0':'default','1':'default','2':'optimistic','3':'optimistic',
                '4':'market_maker','5':'market_maker','6':'speculator','7':'speculator'}
    logger=logging.getLogger('replay')
    agents=[]
    for aid in AGENT_IDS:
        special=ap.get('type_specific_params',{}).get(role_by_id[aid],{})
        agent=CachedResponseAgent(aid,
            initial_cash=special.get('initial_cash',ap['initial_cash']),
            initial_shares=special.get('initial_shares',ap['initial_shares']),
            position_limit=special.get('position_limit',ap['position_limit']),
            allow_short_selling=ap['allow_short_selling'],
            margin_requirement=ap['margin_requirement'],margin_base=ap['margin_base'],
            initial_price=controls['initial_price'])
        agent.response=deepcopy(variant['responses'][aid]);agent.replay_count=0
        agents.append(agent)
    ar=AgentRepository(agents,logger,context,
                       borrowing_repository=BorrowingRepository(int(controls['lendable_shares']),True,logger))
    orepo=OrderRepository(logger);book=OrderBook(context,orepo,logger)
    SharedServiceFactory.reset();SharedServiceFactory.initialize(order_book=book,transaction_cost=controls['transaction_cost'])
    sm,execution=OrderServiceFactory.create_services(orepo,ar,book,logger,max_order_size=int(ap['max_order_size']))
    manager=AgentManager(ar,context,None,logger,logger,orepo,sm,book,
                          commitment_calculator=SharedServiceFactory.get_commitment_calculator(),
                          position_calculator=SharedServiceFactory.get_position_calculator())
    decisions=AgentDecisionService(ar,orepo,sm,logger,logger,context,max_concurrency=8)
    engine=MatchingEngine(book,manager,ar,orepo,context,sm,logger,logger,execution)
    recorder=object.__new__(DataRecorder)
    recorder.context=context;recorder.order_data=[];recorder.trade_data=[];recorder.market_data=[]
    name=f"{variant['arm']}_r{variant['repeat']}_"+('verification' if verification else 'main')
    audit=LifecycleAudit(context,ar,orepo,output/'runs'/name,run_id=output.name+':'+name)
    audit.attach(decisions,sm,execution,engine,recorder)
    initial_resources=audit.all_resources()
    initial_resources_sha=digest(initial_resources)

    # Fix collection/validation order explicitly, instead of relying on a seed.
    ar.get_shuffled_agent_ids=lambda:list(bundle['agent_processing_order'])
    original_create=decisions.create_order
    order_indices=Counter()
    ranks={aid:i for i,aid in enumerate(bundle['agent_processing_order'])}
    def create_with_fixed_priority(details,agent_id):
        aid=str(agent_id);index=order_indices[aid];order_indices[aid]+=1
        assert index<1000
        order=original_create(details,agent_id)
        order.replay_priority_key=(ranks[aid],index)
        # Set before creating book entries. Timestamps are experimental priority labels.
        order.timestamp=1_000_000_000.0+ranks[aid]*1000+index
        audit.emit('REPLAY_PRIORITY_ASSIGNED',order,priority_key=list(order.replay_priority_key),
                    priority_timestamp=order.timestamp)
        return order
    decisions.create_order=create_with_fixed_priority
    def fixed_split(orders):
        ordered=sorted(orders,key=lambda order:order.replay_priority_key)
        market=[order for order in ordered if order.order_type=='market']
        limit=[order for order in ordered if order.order_type=='limit']
        audit.emit('ORDER_PROCESSING_SEQUENCE',market_order_ids=[o.order_id for o in market],
                    limit_order_ids=[o.order_id for o in limit],policy='fixed priority; no shuffle')
        return market,limit
    engine.order_processing_service.split_orders_by_type=fixed_split

    pre_state={'price':context.current_price,'best_bid':None,'best_ask':None,'market_depth':{}}
    new_orders=decisions.collect_decisions(pre_state,[],0)
    result=engine.match_orders(new_orders,context.current_price,0)
    context.current_price=result.price
    stamp=datetime.now(timezone.utc).isoformat()
    recorder._record_order_data(0,new_orders,stamp)
    recorder._record_trade_data(0,result.trades,stamp)
    recorder._record_market_data(0,pre_state,result.trades,result.volume,stamp)
    audit.snapshot();rows=audit.export(recorder)
    write_csv(audit.output_dir/'recorded_market.csv',recorder.market_data)
    actual_responses={row['agent_id']:row['response'] for row in audit.decisions}
    assert len(actual_responses)==8 and sum(a.replay_count for a in agents)==8
    for aid in OTHER_IDS:
        assert digest(actual_responses[aid])==digest(bundle['frozen_other_responses'][aid])
    assert digest({aid:actual_responses[aid] for aid in OTHER_IDS})==bundle['frozen_other_response_sha256']

    order_key={str(order.order_id):list(order.replay_priority_key) for order in orepo.orders.values()}
    priorities=[{'agent_id':str(order.agent_id),'priority_key':list(order.replay_priority_key),
                 'priority_timestamp':order.timestamp,'order_id':str(order.order_id)}
                for order in orepo.orders.values()]
    write_csv(audit.output_dir/'order_priorities.csv',priorities)
    fixed_other_priorities=[{key:row[key] for key in ('agent_id','priority_key','priority_timestamp')}
                             for row in priorities if row['agent_id'] in OTHER_IDS]
    maker_rows=[row for row in rows if row['agent_id'] in MAKER_IDS]
    accepted=[row for row in rows if row['accepted']]
    maker_proposed_both=sum({order['decision'].lower() for order in variant['responses'][aid]['orders']}=={'buy','sell'}
                             for aid in MAKER_IDS)
    maker_accepted_both=sum({row['side'] for row in maker_rows if row['agent_id']==aid and row['accepted']}=={'buy','sell'}
                             for aid in MAKER_IDS)
    accepted_qty=sum(float(row['accepted_quantity']) for row in accepted)
    bid_trades=[trade for trade in audit.trades if trade['buyer_id'] in MAKER_IDS]
    status_counts=Counter(row['execution_status'] for row in rows)
    metric={
        'arm':variant['arm'],'repeat':variant['repeat'],'source_mode':bundle['source_mode'],
        'target_source_run':variant.get('target_source_run','new_model_response'),
        'proposed_orders':len(rows),'accepted_orders':len(accepted),
        'rejected_orders':len(rows)-len(accepted),
        'oversized_rejections':sum(str(row['validation_reason']).startswith('MAX_ORDER_SIZE_EXCEEDED') for row in rows),
        'maker_decisions':2,'maker_proposed_two_sided':maker_proposed_both,
        'maker_accepted_two_sided':maker_accepted_both,
        'maker_accepted_buys':sum(row['accepted'] and row['side']=='buy' for row in maker_rows),
        'maker_accepted_sells':sum(row['accepted'] and row['side']=='sell' for row in maker_rows),
        'trade_count':len(audit.trades),'executed_shares':sum(float(t['quantity']) for t in audit.trades),
        'trades_using_maker_bids':len(bid_trades),
        'shares_using_maker_bids':sum(float(t['quantity']) for t in bid_trades),
        'accepted_quantity':accepted_qty,
        'quantity_fill_rate':2*float(result.volume)/accepted_qty if accepted_qty else 0,
        'fully_filled_orders':status_counts['fully_filled'],
        'partially_filled_orders':status_counts['partially_filled'],
        'accepted_unfilled_orders':status_counts['unfilled'],
        'recorded_price':result.price,
        'last_execution_this_round':audit.trades[-1]['price'] if audit.trades else None,
        'post_best_bid':book.get_best_bid(),'post_best_ask':book.get_best_ask(),
        'post_midpoint':book.get_midpoint(),
        'frozen_other_response_sha256':bundle['frozen_other_response_sha256'],
        'initial_resources_sha256':initial_resources_sha,
        'fixed_other_priority_sha256':digest(fixed_other_priorities),
    }
    # Compare replays by stable priority keys, never UUIDs or wall-clock times.
    normalized={
        'initial_resources':initial_resources,
        'orders':[dict(priority=order_key[row['order_id']],agent_id=row['agent_id'],side=row['side'],
                        requested=row['requested_quantity'],accepted=row['accepted'],
                        accepted_qty=row['accepted_quantity'],filled=row['executed_quantity'],
                        remaining=row['remaining_quantity_raw'],removed=row['post_acceptance_removed_quantity'],
                        state=row['final_state'],cash_commitment=row['cash_commitment_final'],
                        share_commitment=row['share_commitment_final']) for row in rows],
        'trades':[dict(buyer_priority=order_key[t['buyer_order_id']],seller_priority=order_key[t['seller_order_id']],
                       quantity=t['quantity'],price=t['price']) for t in audit.trades],
        'final_resources':audit.all_resources(),'recorded_price':result.price,
        'best_bid':book.get_best_bid(),'best_ask':book.get_best_ask(),
    }
    metric['replay_outcome_sha256']=digest(normalized)
    save_json(audit.output_dir/'run_metrics.json',metric)
    save_json(audit.output_dir/'normalized_outcome.json',normalized)
    return metric,rows,audit.trades,audit.events


metrics=[];check_rows=[];all_orders=[];all_trades=[];all_events=[]
for variant in bundle['variants']:
    main,orders,trades,events=replay_one(variant,False)
    again,_,_,_=replay_one(variant,True)
    assert main['replay_outcome_sha256']==again['replay_outcome_sha256']
    assert main['frozen_other_response_sha256']==again['frozen_other_response_sha256']
    metrics.append(main);all_orders.extend(orders);all_trades.extend(trades);all_events.extend(events)
    check_rows.append({'arm':variant['arm'],'repeat':variant['repeat'],
                       'frozen_other_responses':'PASS','fixed_other_priorities':'PASS',
                       'repeat_reproducibility':'PASS','order_trade_resource_linkage':'PASS'})
    print(f"{variant['arm']} r{variant['repeat']}: PASS, trades={main['trade_count']}, MM-bid trades={main['trades_using_maker_bids']}",flush=True)
assert len({m['initial_resources_sha256'] for m in metrics})==1
assert len({m['fixed_other_priority_sha256'] for m in metrics})==1
assert len({m['frozen_other_response_sha256'] for m in metrics})==1
assert not network_attempts
aggregate=[]
for arm in ARMS:
    group=[row for row in metrics if row['arm']==arm]
    volume=sum(row['executed_shares'] for row in group)
    accepted_qty=sum(row['accepted_quantity'] for row in group)
    aggregate.append({
        'arm':arm,'profiles':len(group),'maker_decisions':sum(row['maker_decisions'] for row in group),
        'maker_proposed_two_sided':sum(row['maker_proposed_two_sided'] for row in group),
        'maker_accepted_two_sided':sum(row['maker_accepted_two_sided'] for row in group),
        'accepted_orders':sum(row['accepted_orders'] for row in group),
        'rejected_orders':sum(row['rejected_orders'] for row in group),
        'oversized_rejections':sum(row['oversized_rejections'] for row in group),
        'runs_with_trades':sum(row['trade_count']>0 for row in group),
        'trade_count':sum(row['trade_count'] for row in group),
        'executed_shares':volume,
        'trades_using_maker_bids':sum(row['trades_using_maker_bids'] for row in group),
        'shares_using_maker_bids':sum(row['shares_using_maker_bids'] for row in group),
        'quantity_fill_rate':2*volume/accepted_qty if accepted_qty else 0,
    })
write_csv(output/'replay_runs.csv',metrics)
write_csv(output/'replay_aggregate.csv',aggregate)
write_csv(output/'replay_checks.csv',check_rows)
write_csv(output/'all_order_lifecycle.csv',all_orders)
write_csv(output/'all_trades_linked.csv',all_trades)
with (output/'all_order_events.jsonl').open('w',encoding='utf-8') as stream:
    for row in all_events:stream.write(json.dumps(clean(row),ensure_ascii=False)+'\n')
summary={
    'source_mode':bundle['source_mode'],'reference_run':bundle['reference_run'],
    'main_replays':len(metrics),'verification_replays':len(metrics),
    'frozen_nonmaker_agents':6,'target_market_makers':2,
    'main_replayed_decisions':8*len(metrics),'verification_replayed_decisions':8*len(metrics),
    'main_proposed_orders':len(all_orders),'main_trades':len(all_trades),
    'main_executed_shares':sum(float(t['quantity']) for t in all_trades),
    'replay_worker_model_calls':0,'network_attempts':len(network_attempts),
    'new_generation_requests_used_for_bundle':bundle.get('new_generation_requests_used',0),
    'all_control_checks':'PASS','repeated_outcomes_identical':True,
    'historical_action_normalization':bundle['normalization'],
    'inference_scope':bundle['evidence_scope'],
}
save_json(output/'step3_summary.json',summary)
print('SUMMARY '+json.dumps(summary,ensure_ascii=True),flush=True)
""",
'scripts/generate_step3_mm_responses.py': r"""'''Optional, explicitly enabled model generation; the default notebook mode never calls this runner.'''
from __future__ import annotations
import argparse
import json
import os
import random
import sys
import uuid
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace


def normalize_parsed_response(parsed):
    from agents.agents_api import TradeDecision
    raw = parsed.model_dump(mode='json')
    validated = TradeDecision.model_validate(raw)
    response = validated.model_dump(mode='json')
    response['response_source'] = 'new_market_maker_api_response'
    return raw, response


def generate_from_plan(plan, bundle, cache_dir, client, schema, *, max_new_requests):
    from step3_replay_common import digest, save_json
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    assert digest(bundle['frozen_other_responses']) == plan['frozen_other_response_sha256']
    attempts_path = cache_dir / 'request_attempts.jsonl'
    def attempt_event(attempt_id, request, status):
        with attempts_path.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps({'attempt_id':attempt_id, 'request_sha256':request['request_sha256'],
                                      'request_key':request['request_key'], 'status':status,
                                      'timestamp_utc':datetime.now(timezone.utc).isoformat()}) + '\n')
    requests = list(plan['requests'])
    random.Random(42).shuffle(requests)
    save_json(cache_dir / 'request_schedule.json', {'request_keys': [r['request_key'] for r in requests]})
    complete = {}
    for request in requests:
        cache_path = cache_dir / (request['request_sha256'] + '.json')
        if cache_path.exists():
            saved = json.loads(cache_path.read_text(encoding='utf-8'))
            if saved.get('status') == 'complete' and saved.get('request_sha256') == request['request_sha256']:
                complete[request['request_key']] = saved
    missing = len(requests) - len(complete)
    if missing > max_new_requests:
        raise RuntimeError(f'Requires {missing} new requests; configured maximum is {max_new_requests}')
    new_requests = 0
    for request in requests:
        if request['request_key'] in complete:
            continue
        if client is None:
            raise RuntimeError('A configured API client is required for uncached requests')
        new_requests += 1
        attempt_id = uuid.uuid4().hex
        attempt_event(attempt_id, request, 'started')
        cache_path = cache_dir / (request['request_sha256'] + '.json')
        save_json(cache_path, {'status': 'started', 'request_sha256': request['request_sha256'],
                               'request': request, 'new_requests_for_entry': 1})
        try:
            parse = getattr(client.chat.completions, 'parse', None)
            if parse is None:
                parse = client.beta.chat.completions.parse
            result = parse(
                model=request['model'],
                messages=[{'role':'system','content':request['system_prompt']},
                          {'role':'user','content':request['user_prompt']}],
                response_format=schema, temperature=request['temperature'], seed=request['seed'],
                max_completion_tokens=request['max_completion_tokens'],
            )
            message = result.choices[0].message
            if getattr(message, 'refusal', None) or message.parsed is None:
                raise RuntimeError('The response was refused or did not contain a parsed decision')
            raw, response = normalize_parsed_response(message.parsed)
            saved = {
                'status':'complete','request_sha256':request['request_sha256'],'request':request,
                'raw_parsed_response':raw,'response':response,
                'completion_id':getattr(result,'id',None),'returned_model':getattr(result,'model',None),
                'system_fingerprint':getattr(result,'system_fingerprint',None),
                'raw_content':message.content,
                'usage':result.usage.model_dump() if getattr(result,'usage',None) else None,
                'new_requests_for_entry':1,
            }
            save_json(cache_path, saved)
            complete[request['request_key']] = saved
            attempt_event(attempt_id, request, 'complete')
            print(request['request_key'] + ': cached', flush=True)
        except Exception as error:
            attempt_event(attempt_id, request, 'failed')
            # Store error class/status only; provider exception text can echo credentials.
            save_json(cache_path, {'status':'failed','request_sha256':request['request_sha256'],
                                   'error_type':type(error).__name__,
                                   'http_status':getattr(error,'status_code',None),
                                   'request':request,'new_requests_for_entry':1})
            raise RuntimeError(f"Request {request['request_key']} failed: {type(error).__name__}; "
                               'generation stopped without an automatic retry. Check endpoint/model access.') from None
    assert len(complete) == len(requests)
    result_bundle = deepcopy(bundle)
    result_bundle['source_mode'] = 'new_mm_api'
    for variant in result_bundle['variants']:
        for aid in result_bundle['maker_ids']:
            key = f"{variant['arm']}:r{variant['repeat']}:agent{aid}"
            variant['responses'][aid] = deepcopy(complete[key]['response'])
        variant['target_source_run'] = 'new_api_response'
        variant['target_response_sha256'] = digest({aid:variant['responses'][aid] for aid in result_bundle['maker_ids']})
    result_bundle['normalization']['target_responses'] = 'New complete responses; project validation preserves their Add/Cancel/Replace semantics.'
    result_bundle['evidence_scope'] = ('Fresh market-maker responses under three system prompts; fixed cached nonmaker orders, '
                                        'fixed initial state and processing priorities; three draws per condition do not establish a general market effect.')
    request_hashes = {r['request_sha256'] for r in requests}
    prior_attempts = [json.loads(line) for line in attempts_path.read_text(encoding='utf-8').splitlines()] if attempts_path.exists() else []
    total_attempts = sum(row['status']=='started' and row['request_sha256'] in request_hashes for row in prior_attempts)
    result_bundle['new_generation_requests_used'] = total_attempts
    result_bundle['new_generation_requests_this_execution'] = new_requests
    return result_bundle, {'new_requests_this_execution':new_requests,'total_request_attempts_recorded':total_attempts,
                           'completed_response_cache_entries':len(complete),'planned_requests':len(requests)}


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--source',type=Path,required=True)
    parser.add_argument('--plan',type=Path,required=True)
    parser.add_argument('--cached-bundle',type=Path,required=True)
    parser.add_argument('--cache-dir',type=Path,required=True)
    parser.add_argument('--output-bundle',type=Path,required=True)
    parser.add_argument('--max-new-requests',type=int,default=18)
    parser.add_argument('--base-url',default='')
    parser.add_argument('--allow-api',action='store_true')
    args=parser.parse_args()
    if not args.allow_api:
        raise RuntimeError('API generation is disabled; use the cached-order replay mode or explicitly enable API mode')
    sys.path.insert(0,str(args.source.resolve()/'src'))
    from step3_replay_common import save_json
    from agents.LLMs.services.schema_features import FeatureRegistry
    # This schema import has no client initialization and makes no network calls.
    schema=FeatureRegistry.get_schema_for_features(set(),is_multi_stock=False)
    plan=json.loads(args.plan.read_text(encoding='utf-8'))
    bundle=json.loads(args.cached_bundle.read_text(encoding='utf-8'))
    if args.base_url and args.base_url.rstrip('/') != plan['api_base_url'].rstrip('/'):
        raise RuntimeError('Endpoint differs from the saved plan; prepare a new request plan first')
    missing=0
    for request in plan['requests']:
        path=args.cache_dir/(request['request_sha256']+'.json')
        saved=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
        if saved.get('status')!='complete' or saved.get('request_sha256')!=request['request_sha256']:
            missing+=1
    if missing>args.max_new_requests:
        raise RuntimeError(f'{missing} requests exceed the configured maximum')
    client=None
    if missing:
        from openai import OpenAI
        key=os.environ.get('AHE_STEP3_API_KEY')
        if not key:
            raise RuntimeError('AHE_STEP3_API_KEY is not set for explicitly enabled API generation')
        kwargs={'api_key':key,'max_retries':0,'timeout':60.0,'base_url':plan['api_base_url']}
        client=OpenAI(**kwargs)
    result,stats=generate_from_plan(plan,bundle,args.cache_dir,client,schema,max_new_requests=args.max_new_requests)
    save_json(args.output_bundle,result)
    save_json(args.output_bundle.with_name('generation_summary.json'),stats)
    print('GENERATION '+json.dumps(stats),flush=True)


if __name__=='__main__':
    main()
"""
}
if not REPLAY_ROOT.exists():shutil.copytree(SOURCE_LOGGED,REPLAY_ROOT)
for relative,expected in EXPECTED_SOURCE_HASHES.items():
    text=(REPLAY_ROOT/relative).read_text(encoding="utf-8")
    if hashlib.sha256(text.encode()).hexdigest()!=expected:raise RuntimeError(f"Base file in the Step 3 copy contains other changes: {relative}")
for relative,text in NEW_FILES.items():
    target=REPLAY_ROOT/relative
    if target.exists() and target.read_text(encoding="utf-8")!=text:raise RuntimeError(f"File contains other changes; overwrite stopped: {relative}")
    ast.parse(text,filename=relative)
for relative,text in NEW_FILES.items():(REPLAY_ROOT/relative).write_text(text,encoding="utf-8")
RUNNER=REPLAY_ROOT/"scripts/run_step3_controlled_replay.py"
GENERATOR=REPLAY_ROOT/"scripts/generate_step3_mm_responses.py"
print("All three complete code files are ready and passed syntax checks.")
print("Code copy: ",REPLAY_ROOT)


## Code cell 4/9: Prepare controlled inputs and the request plan

Fix the following:

- Take all 6 non-market-maker responses from `prompt_probe_r1_current`; keep the initial market and accounts identical.
- Process agents 0–7 in order and each agent's orders in their original list order; disable pre-matching shuffling.
- Use timestamps as explicit priority labels determined by agent and within-agent order index, not as reconstructed historical arrival times.
- Enforce Step 1's 1000-share cap and Step 2's observer in every condition.

The historical decision CSV omitted Add/Cancel/Replace. Cached replay therefore normalizes actions to Add and supports **only one round with an initially empty book**. This is an order-replay protocol rather than an exact recovery of complete historical responses.


In [ ]:
prepare=subprocess.run([str(TEST_PYTHON),str(RUNNER),"prepare","--source",str(REPLAY_ROOT),
                        "--output",str(PROTOCOL_DIR),"--model",API_MODEL,"--base-url",API_BASE_URL,"--repeats",str(REPEATS)],
                       cwd=REPLAY_ROOT,capture_output=True,text=True,encoding="utf-8",env=child_environment(),timeout=60)
if prepare.returncode:raise RuntimeError(prepare.stdout[-2000:]+prepare.stderr[-5000:])
CACHED_BUNDLE=PROTOCOL_DIR/"cached_bundle.json"
API_PLAN=PROTOCOL_DIR/"api_plan.json"
protocol=json.loads((PROTOCOL_DIR/"protocol_summary.json").read_text(encoding="utf-8"))
print("Cached profiles: ",protocol["variants"],"; frozen non-makers: ",protocol["frozen_agents"],"; target makers: ",protocol["target_agents"])
print("Model API calls in this cell: 0.")
print("Planned requests if API mode is enabled: ",protocol["planned_new_model_requests_for_api_mode"])
print("Cached-action convention: ",protocol["normalization"]["replace_decision"],", with an initially empty order book.")


## Code cell 5/9: Select existing orders or generate new maker responses

**Default `cached_orders`: select existing orders without reading a key or calling an API.**

If `new_mm_api` was selected in cell 1:
1. Reuse completed response caches with matching request hashes.
2. Call the model for uncached market-maker requests; a fresh default plan has 18 requests.
3. Save actual prompts, complete parsed responses, token usage, returned models, and request-attempt records.
4. Stop on failure without automatic retries or model substitution.

API mode uses the three existing system prompts and a fixed user prompt for the makers. Obtain the key from `OPENAI_API_KEY` or masked input and exclude it from the result archive. Availability depends on the account/team endpoint. Generation can take time; display progress after each response is cached.


In [ ]:
API_CACHE_DIR=WORK_DIR/"model_response_cache"
GENERATION_STATS={"new_requests_this_execution":0,"completed_response_cache_entries":0}
if RUN_MODE=="cached_orders":
    SELECTED_BUNDLE=CACHED_BUNDLE
    print("Existing-order replay selected; new model API calls: 0.")
    print("This mode checks order mechanisms and does not constitute a new model prompt-response experiment.")
else:
    from getpass import getpass
    API_CACHE_DIR.mkdir(exist_ok=True)
    plan=json.loads(API_PLAN.read_text(encoding="utf-8"))
    missing=[]
    for request in plan["requests"]:
        cache_path=API_CACHE_DIR/(request["request_sha256"]+".json")
        saved=json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
        if saved.get("status")!="complete" or saved.get("request_sha256")!=request["request_sha256"]:missing.append(request)
    if len(missing)>MAX_NEW_API_REQUESTS:raise RuntimeError(f"Requires {len(missing)} new requests, exceeding MAX_NEW_API_REQUESTS.")
    print("New API requests needed in this execution: ",len(missing))
    env=child_environment()
    if missing:
        key=os.environ.get("OPENAI_API_KEY") or getpass("Enter a usable API key (masked and not saved): ")
        if not key:raise RuntimeError("API mode requires a usable key; select cached_orders for replay preparation instead.")
        env['AHE_STEP3_API_KEY']=key
        del key
    SELECTED_BUNDLE=SESSION_DIR/"new_mm_bundle.json"
    command=[str(TEST_PYTHON),str(GENERATOR),"--source",str(REPLAY_ROOT),"--plan",str(API_PLAN),
             "--cached-bundle",str(CACHED_BUNDLE),"--cache-dir",str(API_CACHE_DIR),
             "--output-bundle",str(SELECTED_BUNDLE),"--max-new-requests",str(MAX_NEW_API_REQUESTS),"--allow-api"]
    if API_BASE_URL:command += ["--base-url",API_BASE_URL]
    process=subprocess.Popen(command,cwd=REPLAY_ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,encoding="utf-8",env=env)
    env.pop('AHE_STEP3_API_KEY',None)
    for line in process.stdout:print(line.rstrip(),flush=True)
    if process.wait()!=0:raise RuntimeError("Maker-response generation did not complete. Completed caches are retained. Check model, endpoint, and account access before continuing.")
    GENERATION_STATS=json.loads((SESSION_DIR/"generation_summary.json").read_text(encoding="utf-8"))
    print("Actual new API requests in this execution: ",GENERATION_STATS["new_requests_this_execution"])


## Code cell 6/9: Fix inputs and priorities, then run one-round comparisons

Three conditions with three target profiles/response repetitions give 9 main replays. Repeat each once to check reproducibility, for 18 engine runs in total. **The replay worker blocks network access and makes 0 model API calls.** Any API-mode generation took place in the previous cell.


In [ ]:
RUN_OUTPUT_DIR=SESSION_DIR/"replay_results"
process=subprocess.run([str(TEST_PYTHON),str(RUNNER),"replay","--source",str(REPLAY_ROOT),
                        "--output",str(RUN_OUTPUT_DIR),"--bundle",str(SELECTED_BUNDLE)],
                       cwd=REPLAY_ROOT,capture_output=True,text=True,encoding="utf-8",env=child_environment(),timeout=180)
if process.returncode:raise RuntimeError(process.stdout[-3000:]+process.stderr[-7000:])
print("\n".join(line for line in process.stdout.splitlines() if not line.startswith("SUMMARY ")))
SUMMARY=json.loads((RUN_OUTPUT_DIR/"step3_summary.json").read_text(encoding="utf-8"))
print("Main replays: ",SUMMARY["main_replays"],"; verification replays: ",SUMMARY["verification_replays"])
print("Replay-worker model calls: ",SUMMARY["replay_worker_model_calls"])
print("New generation requests in this execution: ",GENERATION_STATS["new_requests_this_execution"])


## Code cell 7/9: Check controls, order linkage, and grouped results

Compare the proposal of two-sided quotes, acceptance of both sides, and actual trades using maker bids separately. These are different stages.

In default mode, the condition labels identify **which existing prompt experiment supplied the orders**, not newly generated model results. Continue to distinguish executions, recorded prices, and no-trade quote fallback.


In [ ]:
runs=read_csv(RUN_OUTPUT_DIR/"replay_runs.csv")
checks=read_csv(RUN_OUTPUT_DIR/"replay_checks.csv")
aggregate=read_csv(RUN_OUTPUT_DIR/"replay_aggregate.csv")
orders=read_csv(RUN_OUTPUT_DIR/"all_order_lifecycle.csv")
trades=read_csv(RUN_OUTPUT_DIR/"all_trades_linked.csv")
selected=json.loads(SELECTED_BUNDLE.read_text(encoding="utf-8"))
assert len(runs)==len(checks)==3*REPEATS
for row in checks:
    assert all(row[key]=="PASS" for key in ["frozen_other_responses","fixed_other_priorities","repeat_reproducibility","order_trade_resource_linkage"])
for key in ["frozen_other_response_sha256","initial_resources_sha256","fixed_other_priority_sha256"]:
    assert len({row[key] for row in runs})==1
by_order={row["order_id"]:row for row in orders}
assert len(by_order)==len(orders)
filled=defaultdict(float)
for trade in trades:
    for field in ("buyer_order_id","seller_order_id"):
        oid=trade[field];assert oid in by_order and by_order[oid]["accepted"]=="True"
        filled[oid]+=float(trade["quantity"])
for row in orders:
    assert abs(float(row["executed_quantity"])-filled[row["order_id"]])<1e-8
    if row["accepted"]=="True":
        assert float(row["accepted_quantity"])<=1000
        assert abs(float(row["accepted_quantity"])-float(row["executed_quantity"])-float(row["remaining_quantity_raw"])-float(row["post_acceptance_removed_quantity"]))<1e-8
assert SUMMARY["replay_worker_model_calls"]==0 and SUMMARY["network_attempts"]==0
print("Frozen inputs, priorities, repeated replay, and order-level linkage: all PASS.")
labels={"current":"Current","earliest_public":"Earliest public","mandatory_two_sided":"Mandatory two-sided"}
view=[]
for row in aggregate:
    view.append({"arm":labels[row["arm"]],"proposed":row["maker_proposed_two_sided"]+"/"+row["maker_decisions"],
                 "accepted":row["maker_accepted_two_sided"]+"/"+row["maker_decisions"],
                 "trades":row["trade_count"],"bidtrades":row["trades_using_maker_bids"],
                 "volume":row["executed_shares"],"oversized":row["oversized_rejections"]})
display_table(view,[("arm","Maker condition"),("proposed","Two-sided quotes proposed"),("accepted","Both sides accepted"),
                    ("trades","Actual trades"),("bidtrades","Trades using maker bids"),
                    ("volume","Executed shares"),("oversized","Oversized orders rejected")])
if RUN_MODE=="cached_orders":
    print("Scope: compare validation and execution of recorded maker orders against the same counterparty orders, initial state, and priorities.")
    print("These results do not separately identify how a new prompt changes model decisions and are not directly equivalent to the original nine uncontrolled probes.")
else:
    print("Scope: controlled comparison of fresh maker responses under three prompts, with other agents' cached orders fixed.")
    print("There are only three response repetitions per condition. Model outputs may vary, and long-term market effects have not been tested.")


## Code cell 8/9: Trace the execution of a maker buy order

Inspect the first Mandatory two-sided main replay by default. Display the selected order's own events and retain both legs on whole-trade events; do not mix the counterparty's state rows into the selected order's state trace.


In [ ]:
case=RUN_OUTPUT_DIR/"runs/mandatory_two_sided_r1_main"
case_orders=read_csv(case/"order_lifecycle.csv")
candidates=[row for row in case_orders if row["agent_id"] in ("4","5") and row["side"]=="buy"]
if not candidates:
    print("This response profile contains no maker buy order; retain that outcome as part of the experiment.")
else:
    target=candidates[0]
    case_events=read_jsonl(case/"order_events.jsonl")
    types={"ORDER_REGISTERED","REPLAY_PRIORITY_ASSIGNED","VALIDATION_ACCEPTED","VALIDATION_REJECTED",           "STATE_CHANGED","TRADE_EXECUTION_STARTED","TRADE_EXECUTED","RESOURCE_RELEASED","POSITION_UPDATED"}
    trace=[]
    for event in case_events:
        trade_event=event["event_type"] in ("TRADE_EXECUTION_STARTED","TRADE_EXECUTED")
        related=event["order_id"]==target["order_id"] or (trade_event and target["order_id"] in (event["buyer_order_id"],event["seller_order_id"]))
        if not related or event["event_type"] not in types:continue
        snapshot=event.get("order") or {}
        trace.append({"seq":event["event_seq"],"owner":snapshot.get("agent_id","Both counterparties" if trade_event else ""),
                      "event":event["event_type"],"state":snapshot.get("state",""),
                      "filled":snapshot.get("filled_quantity",""),"remaining":snapshot.get("remaining_quantity","")})
    assert all(row["owner"] in (target["agent_id"],"Both counterparties") for row in trace)
    display_table(trace,[("seq","Event sequence"),("owner","Actor"),("event","Event"),("state","State"),("filled","Cumulative execution"),("remaining","Remaining quantity")])
    print("Requested: ",target["requested_quantity"],"; accepted: ",target["accepted_quantity"],"; executed: ",target["executed_quantity"])
    print("Final execution status: ",target["execution_status"],"; state: ",target["final_state"])


## Code cell 9/9: Save results and the reportable scope

Save the protocol, actual orders and trades, complete event stream, and added code. API mode also saves response caches and request-attempt records. Exclude credentials and the entire Python environment.


In [ ]:
import zipfile
if RUN_MODE=="cached_orders":
    progress=("This stage established and validated one-round replay with fixed non-maker orders, initial accounts, and processing priorities. "
              "Three existing maker-order profiles per condition produced nine main replays and nine reproducibility checks; identical inputs gave identical outcomes. "
              "The results examine how recorded orders become accepted and executed orders, rather than a new model prompt-response experiment. "
              "Historical CSV files omit replacement actions, so initial-empty-book replay normalizes actions to Add; the 1000-share cap is enforced in all conditions.")
else:
    progress=("This stage fixed six non-makers' cached orders, initial accounts, and processing priorities, "
              "obtained new responses from two makers under the three existing prompts, and compared two-sided quoting, validation, and actual execution. "
              "Model requests, complete responses, and trade links are recorded. Findings are limited to one round and a small number of response repetitions; long-term effects remain untested.")
(SESSION_DIR/"progress_note_cn.txt").write_text(progress,encoding="utf-8")
metadata={"mode":RUN_MODE,"repeats":REPEATS,"api_model":API_MODEL,
          "new_requests_this_execution":GENERATION_STATS["new_requests_this_execution"],
          "replay_summary":SUMMARY,"note":progress}
(SESSION_DIR/"session_summary.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
ZIP_PATH=WORK_DIR/(SESSION_DIR.name+"_Step3_Results.zip")
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(SESSION_DIR.rglob("*")):
        if path.is_file():archive.write(path,arcname="session/"+path.relative_to(SESSION_DIR).as_posix())
    for relative in NEW_FILES:archive.write(REPLAY_ROOT/relative,arcname=relative)
    if RUN_MODE=="new_mm_api":
        plan=json.loads(API_PLAN.read_text(encoding="utf-8"))
        for request in plan["requests"]:
            path=API_CACHE_DIR/(request["request_sha256"]+".json")
            if path.exists():archive.write(path,arcname="model_response_cache/"+path.name)
        for name in ("request_schedule.json","request_attempts.jsonl"):
            path=API_CACHE_DIR/name
            if path.exists():archive.write(path,arcname="model_response_cache/"+name)
print("Run mode: ",RUN_MODE)
print("New model requests in this execution: ",GENERATION_STATS["new_requests_this_execution"])
print("Reportable scope: ",progress)
print("Result archive: ",ZIP_PATH)
print("Summary table: ",RUN_OUTPUT_DIR/"replay_aggregate.csv")
print("Press Ctrl + S to save, then export HTML to retain your execution results.")


### Determine which part is complete

- `cached_orders`: controlled-counterparty/priority replay preparation and mechanism comparison; 0 model API calls.
- `new_mm_api`: new maker responses followed by controlled replay; requires a real model API, while matching caches can avoid repeated requests.
- Identical outcomes from the same responses establish stable engine controls, not identical future model responses.
- This is a one-round diagnostic. Fifteen-round persistence remains the next step.

Retain complete errors. If API mode reports unavailable model access, check the team endpoint and permissions first; a replacement model is a different model condition.
